# MOX crystal properties from `Matrix::setCrystalProperties()`

This notebook reproduces the current SCIANTIX formulas in `src/classes/Matrix.C` for:

- lattice parameter
- theoretical density

as a function of:

- `q = Pu / (U + Pu)`
- `O/M`

Notes:

- For `q <= 0.10`, the current code follows the UO2 branch.
- In that branch, if `chromium_content_ppm == 0`, the function keeps the default UO2 values:
  - `a = 5.47109e-10 m`
  - `rho = 10960 kg/m3`
- For MOX (`q > 0.10`), the code uses the NEA-based correlation currently implemented in SCIANTIX.
- Only hypostoichiometric deviations are used in the MOX branch, so `x_hypo = max(0, 2 - O/M)`.


In [8]:
import math

AVOGADRO = 6.02214076e23
MOLAR_MASS_OXYGEN = 15.999
MOLAR_MASS_CHROMIUM = 51.9961

# SUPERFACT
DEFAULT_U_VECTOR = {
    "U234": 0.0,
    "U235":0.7,
    "U236": 0.0,
    "U237": 0.0,
    "U238": 99.3,
}

DEFAULT_PU_VECTOR = {
    "Pu238": 1.3,
    "Pu239": 60.4,
    "Pu240": 23.4,
    "Pu241": 10.4,
    "Pu242": 4.5,
}


def mean_uranium_molar_mass(u_vector):
    masses = {
        "U234": 234.04095,
        "U235": 235.04393,
        "U236": 236.04557,
        "U237": 237.04817,
        "U238": 238.05079,
    }
    total = sum(u_vector.values())
    if total <= 0.0:
        return 238.02891
    return sum(u_vector[k] * masses[k] for k in masses) / total


def mean_plutonium_molar_mass(pu_vector):
    masses = {
        "Pu238": 238.04956,
        "Pu239": 239.05216,
        "Pu240": 240.05381,
        "Pu241": 241.05685,
        "Pu242": 242.05874,
    }
    total = sum(pu_vector.values())
    if total <= 0.0:
        return 239.05216
    return sum(pu_vector[k] * masses[k] for k in masses) / total


def chromium_mole_fraction(chromium_content_ppm, u_vector):
    if chromium_content_ppm <= 0.0:
        return 0.0
    molar_mass_uranium = mean_uranium_molar_mass(u_vector)
    return (
        chromium_content_ppm * 1.0e-6 * (molar_mass_uranium + 2.0 * MOLAR_MASS_OXYGEN)
        / (
            chromium_content_ppm * 1.0e-6 * (molar_mass_uranium - MOLAR_MASS_CHROMIUM)
            + molar_mass_uranium
        )
    )


def crystal_properties_like_sciantix(
    q,
    om_ratio,
    uranium_vector=None,
    plutonium_vector=None,
    chromium_content_ppm=0.0,
):
    """Reproduce the current SCIANTIX Matrix::setCrystalProperties() implementation."""
    u_vector = dict(DEFAULT_U_VECTOR if uranium_vector is None else uranium_vector)
    pu_vector = dict(DEFAULT_PU_VECTOR if plutonium_vector is None else plutonium_vector)

    lattice_parameter = 5.47109e-10
    matrix_density = 10960.0

    # UO2 / Cr-doped UO2 branch
    if q <= 0.10:
        if chromium_content_ppm > 0.0:
            lattice_parameter_uo2_angstrom = 5.47109
            lattice_parameter = (lattice_parameter_uo2_angstrom - 1.2e-6 * chromium_content_ppm) * 1.0e-10
            x_cr = chromium_mole_fraction(chromium_content_ppm, u_vector)
            molar_mass_uranium = mean_uranium_molar_mass(u_vector)
            formula_unit_mass = (
                (1.0 - x_cr) * molar_mass_uranium + x_cr * MOLAR_MASS_CHROMIUM + 2.0 * MOLAR_MASS_OXYGEN
            )
            matrix_density = 4.0 * formula_unit_mass * 1.0e-3 / (AVOGADRO * lattice_parameter**3)

        return {
            "branch": "UO2",
            "q": q,
            "O/M": om_ratio,
            "x_hypo": max(0.0, 2.0 - om_ratio),
            "lattice_parameter_m": lattice_parameter,
            "lattice_parameter_A": lattice_parameter * 1.0e10,
            "theoretical_density_kg_m3": matrix_density,
        }

    # MOX branch
    x_hypo = max(0.0, 2.0 - om_ratio)

    u_tot = sum(u_vector.values())
    pu_tot = sum(pu_vector.values())
    am_tot = 0.0
    np_tot = 0.0
    hm_tot = u_tot + pu_tot + am_tot + np_tot

    z_pu = pu_tot / hm_tot if hm_tot > 0.0 else 0.0
    y_am = am_tot / hm_tot if hm_tot > 0.0 else 0.0
    y_np = np_tot / hm_tot if hm_tot > 0.0 else 0.0
    y_u = max(0.0, 1.0 - z_pu - y_am - y_np)

    r_u = 0.9972
    r_pu = 0.9642
    r_am = 1.0900
    r_np = 0.9806
    r_o = 1.3720

    r_cation = r_u * y_u + r_pu * z_pu + r_am * y_am + r_np * y_np
    a_angstrom = (4.0 / math.sqrt(3.0)) * (r_cation * (1.0 + 0.112 * x_hypo) + r_o)
    lattice_parameter = a_angstrom * 1.0e-10

    m_u = mean_uranium_molar_mass(u_vector)
    m_pu = mean_plutonium_molar_mass(pu_vector)
    m_am = 243.06138
    m_np = 237.04817

    m_fu = y_u * m_u + z_pu * m_pu + y_am * m_am + y_np * m_np + (2.0 - x_hypo) * MOLAR_MASS_OXYGEN
    matrix_density = (4.0 * m_fu * 1.0e-3) / (AVOGADRO * lattice_parameter**3)

    return {
        "branch": "MOX",
        "q": q,
        "O/M": om_ratio,
        "x_hypo": x_hypo,
        "lattice_parameter_m": lattice_parameter,
        "lattice_parameter_A": a_angstrom,
        "theoretical_density_kg_m3": matrix_density,
        "z_Pu": z_pu,
        "y_U": y_u,
        "M_U_g_mol": m_u,
        "M_Pu_g_mol": m_pu,
        "M_fu_g_mol": m_fu,
    }


In [10]:
# Single evaluation: edit q and O/M here

q = 0.288
om_ratio = 1.975

result = crystal_properties_like_sciantix(q=q, om_ratio=om_ratio)
result

{'branch': 'MOX',
 'q': 0.288,
 'O/M': 1.975,
 'x_hypo': 0.02499999999999991,
 'lattice_parameter_m': 5.439669436270466e-10,
 'lattice_parameter_A': 5.439669436270466,
 'theoretical_density_kg_m3': 11159.19458954297,
 'z_Pu': 0.5,
 'y_U': 0.5,
 'M_U_g_mol': 238.02974198,
 'M_Pu_g_mol': 239.61729615999997,
 'M_fu_g_mol': 270.42154407}